# FIFA World Cup Pipeline Report

This notebook is the single source of truth for the project.

It consolidates extraction, cleaning, EDA, statistical analysis, and final combined CSV validation into one workflow.

Run the cells from top to bottom.


In [25]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 140)
pd.set_option('display.max_colwidth', 120)


def find_project_root(start: Path | None = None) -> Path:
    current = start or Path.cwd()
    for candidate in [current, *current.parents]:
        if (candidate / 'data').exists() and (candidate / 'notebooks').exists():
            return candidate
    return current


PROJECT_ROOT = find_project_root()
BASE = PROJECT_ROOT / 'data'
RAW = BASE / 'raw'
PROCESSED = BASE / 'processed'

RAW_FILES = {
    'WorldCups': RAW / 'WorldCups.csv',
    'WorldCupMatches': RAW / 'WorldCupMatches.csv',
    'WorldCupPlayers': RAW / 'WorldCupPlayers.csv',
}

CLEAN_FILE_MAP = {
    'WorldCups': PROCESSED / 'wc_cups_clean.csv',
    'WorldCupMatches': PROCESSED / 'wc_matches_clean.csv',
    'WorldCupPlayers': PROCESSED / 'wc_players_combined.csv',
}

FINAL_CANDIDATES = [
    PROCESSED / 'WorldCup_Combined.csv',
    PROCESSED / 'wc_players_combined.csv',
]


def load_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, encoding='utf-8')


def file_size_kb(path: Path) -> float:
    return round(path.stat().st_size / 1024, 1) if path.exists() else 0.0


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = (
        out.columns.astype(str)
        .str.strip()
        .str.replace(r'[^\w]+', '_', regex=True)
        .str.replace(r'_+', '_', regex=True)
        .str.strip('_')
    )
    return out


def snapshot(df: pd.DataFrame) -> dict:
    return {
        'rows': len(df),
        'cols': df.shape[1],
        'nulls': int(df.isna().sum().sum()),
        'dupes': int(df.duplicated().sum()),
        'memory_mb': round(df.memory_usage(deep=True).sum() / 1024 ** 2, 2),
    }


def first_existing(df: pd.DataFrame, candidates: list[str]) -> str | None:
    for name in candidates:
        if name in df.columns:
            return name
    return None


def load_optional(path: Path) -> pd.DataFrame | None:
    return load_csv(path) if path.exists() else None


## 1) Extraction metrics

This section summarizes the raw source files before any cleaning or merging.

In [26]:
raw_dfs = {}
raw_rows = []

for name, path in RAW_FILES.items():
    df = load_csv(path)
    raw_dfs[name] = df
    raw_rows.append({
        'dataset': name,
        'file': str(path),
        'rows': len(df),
        'cols': df.shape[1],
        'nulls': int(df.isna().sum().sum()),
        'dupes': int(df.duplicated().sum()),
        'memory_mb': round(df.memory_usage(deep=True).sum() / 1024 ** 2, 2),
        'file_size_kb': file_size_kb(path),
    })

raw_summary = pd.DataFrame(raw_rows)
display(raw_summary)

print('Extraction details:')
print(f"- Total raw rows: {sum(r['rows'] for r in raw_rows):,}")
print(f"- Total raw columns: {sum(r['cols'] for r in raw_rows):,}")
print(f"- Raw files loaded: {len(raw_dfs)}")

for name, df in raw_dfs.items():
    print(f'\n{name} columns:')
    print(list(df.columns))
    null_pct = (df.isna().sum() / len(df) * 100).sort_values(ascending=False)
    top_nulls = null_pct[null_pct > 0].head(10)
    if not top_nulls.empty:
        display(top_nulls.to_frame('null_%'))


,dataset,file,rows,cols,nulls,dupes,memory_mb,file_size_kb
0,WorldCups,/Users/aarsh.user/Desktop/Major Project (for internship)/D_V_A_/SectionE_g11_FIFA/data/raw/WorldCups.csv,20,10,0,0,0.01,1.4
1,WorldCupMatches,/Users/aarsh.user/Desktop/Major Project (for internship)/D_V_A_/SectionE_g11_FIFA/data/raw/WorldCupMatches.csv,4572,20,74402,3735,2.32,233.4
2,WorldCupPlayers,/Users/aarsh.user/Desktop/Major Project (for internship)/D_V_A_/SectionE_g11_FIFA/data/raw/WorldCupPlayers.csv,37784,9,62356,736,13.09,2100.2


Extraction details:
- Total raw rows: 42,376
- Total raw columns: 39
- Raw files loaded: 3

WorldCups columns:
['Year', 'Country', 'Winner', 'Runners-Up', 'Third', 'Fourth', 'GoalsScored', 'QualifiedTeams', 'MatchesPlayed', 'Attendance']

WorldCupMatches columns:
['Year', 'Datetime', 'Stage', 'Stadium', 'City', 'Home Team Name', 'Home Team Goals', 'Away Team Goals', 'Away Team Name', 'Win conditions', 'Attendance', 'Half-time Home Goals', 'Half-time Away Goals', 'Referee', 'Assistant 1', 'Assistant 2', 'RoundID', 'MatchID', 'Home Team Initials', 'Away Team Initials']


,null_%
Attendance,81.408574
Datetime,81.364829
Home Team Initials,81.364829
MatchID,81.364829
RoundID,81.364829
Assistant 2,81.364829
Assistant 1,81.364829
Referee,81.364829
Half-time Away Goals,81.364829
Half-time Home Goals,81.364829



WorldCupPlayers columns:
['RoundID', 'MatchID', 'Team Initials', 'Coach Name', 'Line-up', 'Shirt Number', 'Player Name', 'Position', 'Event']


,null_%
Position,89.035041
Event,75.997777


## 2) Cleaning metrics

This section compares the raw inputs with the cleaned/processed outputs and shows how much the pipeline reduced noise and inconsistency.

## Data Quality Validation

This section runs comprehensive validation checks on all processed datasets before analysis.
Covers: structure, duplicates, nulls, value ranges, key integrity, and domain-specific rules.


In [28]:

# Load all processed datasets for validation
DF  = clean_dfs.get('WorldCupPlayers', load_csv(CLEAN_FILE_MAP['WorldCupPlayers']))
MC  = clean_dfs.get('WorldCupMatches', load_csv(CLEAN_FILE_MAP['WorldCupMatches']))
CU  = clean_dfs.get('WorldCups', load_csv(CLEAN_FILE_MAP['WorldCups']))

PASS = "✓ PASS"
FAIL = "✗ FAIL"

results = []

def record(section, name, passed, detail=""):
    results.append((section, name, passed, detail))
    tag = PASS if passed else FAIL
    print(f"{tag}  {name}" + (f"  →  {detail}" if detail else ""))

def section_header(title):
    print(f"\n{'═'*70}")
    print(f"  {title}")
    print(f"{'═'*70}")

# ══════════════════════════════════════════════════════════════════════════
# 1. STRUCTURE
# ══════════════════════════════════════════════════════════════════════════
section_header("1. STRUCTURE — shape and minimum requirements")

record("Structure", "Players combined ≥ 10,000 rows",
       len(DF) >= 10_000,
       f"{len(DF):,} rows")

record("Structure", "Players combined ≥ 15 columns",
       len(DF.columns) >= 15,
       f"{len(DF.columns)} columns")

record("Structure", "Matches table is 836 rows",
       len(MC) == 836,
       f"{len(MC)} rows")

record("Structure", "Cups table is 20 rows (one per tournament)",
       len(CU) == 20,
       f"{len(CU)} rows")

record("Structure", "Players combined has no completely empty rows",
       DF.isnull().all(axis=1).sum() == 0,
       f"{DF.isnull().all(axis=1).sum()} fully-null rows")

record("Structure", "Matches table has no completely empty rows",
       MC.isnull().all(axis=1).sum() == 0)

record("Structure", "All 20 tournament years present in players data",
       set(CU["Year"]) == set(DF["Year"].unique()),
       f"{len(set(DF['Year'].unique()))} years found")

# ══════════════════════════════════════════════════════════════════════════
# 2. DUPLICATES
# ══════════════════════════════════════════════════════════════════════════
section_header("2. DUPLICATES — check for exact and key duplicates")

exact_dups = DF.duplicated().sum()
record("Duplicates", "No fully identical rows in players combined",
       exact_dups == 0,
       f"{exact_dups} exact duplicate rows")

key_cols = ["MatchID", "Player_Name", "Team_Initials", "Shirt_Number"]
key_cols_exist = [c for c in key_cols if c in DF.columns]
if key_cols_exist:
    key_dups = DF.duplicated(subset=key_cols_exist).sum()
    record("Duplicates", f"No duplicate ({' + '.join(key_cols_exist)})",
           key_dups == 0,
           f"{key_dups} key duplicates")

if "MatchID" in MC.columns:
    mc_dups = MC.duplicated(subset=["MatchID"]).sum()
    record("Duplicates", "No duplicate MatchIDs in matches table",
           mc_dups == 0,
           f"{mc_dups} duplicate MatchIDs")

if "Year" in CU.columns:
    cu_dups = CU.duplicated(subset=["Year"]).sum()
    record("Duplicates", "No duplicate Years in cups table",
           cu_dups == 0,
           f"{cu_dups} duplicate Years")

# ══════════════════════════════════════════════════════════════════════════
# 3. NULL ANALYSIS
# ══════════════════════════════════════════════════════════════════════════
section_header("3. NULL ANALYSIS — only expected nulls allowed")

# Columns that should be 100% populated (core join keys)
must_have_cols = ["Year", "MatchID", "Player_Name", "Team_Initials"]
must_have_cols = [c for c in must_have_cols if c in DF.columns]

for col in must_have_cols:
    nulls_in_col = DF[col].isna().sum()
    record("Nulls", f"Column '{col}' is 100% populated",
           nulls_in_col == 0,
           f"{nulls_in_col} nulls" if nulls_in_col > 0 else "")

# Columns that may have nulls (acceptable)
optional_cols = ["Rest_Days", "Shirt_Number", "Coach", "Event"]
optional_cols = [c for c in optional_cols if c in DF.columns]

print("\nOptional columns (nulls acceptable):")
for col in optional_cols:
    null_pct = DF[col].isna().sum() / len(DF) * 100
    print(f"  {col:<20} {DF[col].isna().sum():>6,} nulls ({null_pct:.1f}%)")

# ══════════════════════════════════════════════════════════════════════════
# 4. VALUE RANGES & CONSISTENCY
# ══════════════════════════════════════════════════════════════════════════
section_header("4. VALUE RANGES & CONSISTENCY — logical bounds")

if "Year" in DF.columns:
    year_min, year_max = DF["Year"].min(), DF["Year"].max()
    record("Range", "Tournament years in valid range [1930-2014]",
           1930 <= year_min and year_max <= 2014,
           f"{year_min}-{year_max}")

if "Goals" in DF.columns:
    goals_min, goals_max = DF["Goals"].min(), DF["Goals"].max()
    record("Range", "Player goals per row in [0-5]",
           goals_min >= 0 and goals_max <= 10,
           f"min={goals_min}, max={goals_max}")

if "Yellow_Cards" in DF.columns and "Red_Cards" in DF.columns:
    yc_max = DF["Yellow_Cards"].max()
    rc_max = DF["Red_Cards"].max()
    record("Range", "Yellow cards per row ≤ 3",
           yc_max <= 3,
           f"max={yc_max}")
    record("Range", "Red cards per row ≤ 1",
           rc_max <= 1,
           f"max={rc_max}")

if "Attendance" in MC.columns:
    att_min, att_max = MC["Attendance"].min(), MC["Attendance"].max()
    record("Range", "Attendance in reasonable range [1,000-200,000]",
           att_min >= 1000 and att_max <= 200000,
           f"min={att_min:,}, max={att_max:,}")

# ══════════════════════════════════════════════════════════════════════════
# 5. JOIN INTEGRITY
# ══════════════════════════════════════════════════════════════════════════
section_header("5. JOIN INTEGRITY — foreign key validation")

if "Year" in DF.columns and "Year" in CU.columns:
    df_years = set(DF["Year"].unique())
    cu_years = set(CU["Year"].unique())
    record("Join", "All player data years exist in tournaments table",
           df_years <= cu_years,
           f"Player years: {len(df_years)}, Cups years: {len(cu_years)}")

if "MatchID" in DF.columns and "MatchID" in MC.columns:
    df_matches = set(DF["MatchID"].dropna().unique())
    mc_matches = set(MC["MatchID"].unique())
    orphaned = len(df_matches - mc_matches)
    record("Join", "All player MatchIDs exist in matches table",
           orphaned == 0,
           f"Orphaned: {orphaned}" if orphaned > 0 else "✓")

if "Team_Initials" in DF.columns and "Home_Initials" in MC.columns:
    df_teams = set(DF["Team_Initials"].dropna().unique())
    mc_home_teams = set(MC["Home_Initials"].dropna().unique())
    mc_away_teams = set(MC["Away_Initials"].dropna().unique())
    mc_all_teams = mc_home_teams | mc_away_teams
    orphaned_teams = len(df_teams - mc_all_teams)
    record("Join", "All player teams exist in matches table",
           orphaned_teams == 0,
           f"Orphaned teams: {orphaned_teams}" if orphaned_teams > 0 else "✓")

# ══════════════════════════════════════════════════════════════════════════
# 6. SUMMARY
# ══════════════════════════════════════════════════════════════════════════
section_header("6. VALIDATION SUMMARY")

passed_count = sum(1 for _, _, p, _ in results if p)
total_count = len(results)
pass_rate = round(passed_count / total_count * 100, 1) if total_count > 0 else 0

print(f"\nTotal checks: {total_count}")
print(f"Passed: {passed_count}")
print(f"Failed: {total_count - passed_count}")
print(f"Pass rate: {pass_rate}%")

if pass_rate == 100:
    print("\n✓ ALL VALIDATION CHECKS PASSED - Data is ready for analysis")
else:
    print(f"\n⚠ {total_count - passed_count} validation checks failed - review above")

# Summary table
results_df = pd.DataFrame(results, columns=['Section', 'Test', 'Passed', 'Detail'])
display(results_df[['Section', 'Test', 'Passed', 'Detail']])



══════════════════════════════════════════════════════════════════════
  1. STRUCTURE — shape and minimum requirements
══════════════════════════════════════════════════════════════════════
✓ PASS  Players combined ≥ 10,000 rows  →  37,048 rows
✓ PASS  Players combined ≥ 15 columns  →  57 columns
✓ PASS  Matches table is 836 rows  →  836 rows
✓ PASS  Cups table is 20 rows (one per tournament)  →  20 rows
✓ PASS  Players combined has no completely empty rows  →  0 fully-null rows
✓ PASS  Matches table has no completely empty rows
✓ PASS  All 20 tournament years present in players data  →  20 years found

══════════════════════════════════════════════════════════════════════
  2. DUPLICATES — check for exact and key duplicates
══════════════════════════════════════════════════════════════════════
✓ PASS  No fully identical rows in players combined  →  0 exact duplicate rows
✓ PASS  No duplicate (MatchID + Player_Name + Team_Initials + Shirt_Number)  →  0 key duplicates
✓ PASS  No duplic

,Section,Test,Passed,Detail
0,Structure,"Players combined ≥ 10,000 rows",True,"37,048 rows"
1,Structure,Players combined ≥ 15 columns,True,57 columns
2,Structure,Matches table is 836 rows,True,836 rows
3,Structure,Cups table is 20 rows (one per tournament),True,20 rows
4,Structure,Players combined has no completely empty rows,True,0 fully-null rows
5,Structure,Matches table has no completely empty rows,True,
6,Structure,All 20 tournament years present in players data,True,20 years found
7,Duplicates,No fully identical rows in players combined,True,0 exact duplicate rows
8,Duplicates,No duplicate (MatchID + Player_Name + Team_Initials + Shirt_Number),True,0 key duplicates
9,Duplicates,No duplicate MatchIDs in matches table,True,0 duplicate MatchIDs


In [27]:
clean_rows = []
clean_dfs = {}

for raw_name, clean_path in CLEAN_FILE_MAP.items():
    if clean_path.exists():
        df = load_csv(clean_path)
        clean_dfs[raw_name] = df
        clean_rows.append({
            'dataset': raw_name,
            'file': str(clean_path),
            'rows': len(df),
            'cols': df.shape[1],
            'nulls': int(df.isna().sum().sum()),
            'dupes': int(df.duplicated().sum()),
            'memory_mb': round(df.memory_usage(deep=True).sum() / 1024 ** 2, 2),
            'file_size_kb': file_size_kb(clean_path),
        })

clean_summary = pd.DataFrame(clean_rows)
display(clean_summary)

if not raw_summary.empty and not clean_summary.empty:
    compare = raw_summary.merge(clean_summary, on='dataset', how='left', suffixes=('_raw', '_clean'))
    compare['row_reduction'] = compare['rows_raw'] - compare['rows_clean']
    compare['row_reduction_%'] = (compare['row_reduction'] / compare['rows_raw'] * 100).round(2)
    compare['null_reduction'] = compare['nulls_raw'] - compare['nulls_clean']
    compare['dup_reduction'] = compare['dupes_raw'] - compare['dupes_clean']
    display(compare[[
        'dataset', 'rows_raw', 'rows_clean', 'row_reduction', 'row_reduction_%',
        'nulls_raw', 'nulls_clean', 'null_reduction', 'dupes_raw', 'dupes_clean', 'dup_reduction'
    ]])

print('Cleaning notes:')
print('- WorldCupMatches is expected to shrink heavily after blank Year rows are removed.')
print('- WorldCupPlayers should stay at player-match grain.')
print('- The combined final file should retain stable join keys with minimal duplicates.')

for name, df in clean_dfs.items():
    print(f'\n{name} cleaned columns:')
    print(list(df.columns))


,dataset,file,rows,cols,nulls,dupes,memory_mb,file_size_kb
0,WorldCups,/Users/aarsh.user/Desktop/Major Project (for internship)/D_V_A_/SectionE_g11_FIFA/data/processed/wc_cups_clean.csv,20,10,0,0,0.01,1.4
1,WorldCupMatches,/Users/aarsh.user/Desktop/Major Project (for internship)/D_V_A_/SectionE_g11_FIFA/data/processed/wc_matches_clean.csv,836,34,425,0,0.90,170.9
2,WorldCupPlayers,/Users/aarsh.user/Desktop/Major Project (for internship)/D_V_A_/SectionE_g11_FIFA/data/processed/wc_players_combine...,37048,57,12436,0,50.18,11540.3


,dataset,rows_raw,rows_clean,row_reduction,row_reduction_%,nulls_raw,nulls_clean,null_reduction,dupes_raw,dupes_clean,dup_reduction
0,WorldCups,20,20,0,0.00,0,0,0,0,0,0
1,WorldCupMatches,4572,836,3736,81.71,74402,425,73977,3735,0,3735
2,WorldCupPlayers,37784,37048,736,1.95,62356,12436,49920,736,0,736


Cleaning notes:
- WorldCupMatches is expected to shrink heavily after blank Year rows are removed.
- WorldCupPlayers should stay at player-match grain.
- The combined final file should retain stable join keys with minimal duplicates.

WorldCups cleaned columns:
['Year', 'Host_Country', 'Winner', 'Runners_Up', 'Third', 'Fourth', 'Tournament_Total_Goals', 'Qualified_Teams', 'Tournament_Matches', 'Tournament_Attendance']

WorldCupMatches cleaned columns:
['Year', 'MatchID', 'RoundID', 'Match_Date', 'Match_Time', 'Day_of_Week', 'Month', 'Stage_Raw', 'Stage_Std', 'Stage_Order', 'Days_Into_Tournament', 'Home_Team', 'Away_Team', 'Home_Initials', 'Away_Initials', 'Tournament_Host', 'Is_Host_Home', 'Is_Host_Away', 'Home_Goals', 'Away_Goals', 'HT_Home_Goals', 'HT_Away_Goals', 'SH_Home_Goals', 'SH_Away_Goals', 'Total_Goals', 'Goal_Diff', 'Match_Result', 'Win_Conditions', 'Attendance', 'Stadium', 'City', 'Referee', 'Rest_Days_Home', 'Rest_Days_Away']

WorldCupPlayers cleaned columns:
['Year', 'Mat

## 3) EDA summary

This section gives high-level descriptive analysis for the combined dataset.

In [19]:
final_path = next((p for p in FINAL_CANDIDATES if p.exists()), None)
if final_path is None:
    raise FileNotFoundError('No combined CSV found. Expected one of: ' + ', '.join(str(p) for p in FINAL_CANDIDATES))

final_df = normalize_columns(load_csv(final_path))
print(f'Final combined file: {final_path}')
print(f'Rows: {len(final_df):,} | Columns: {final_df.shape[1]} | Size: {file_size_kb(final_path)} KB')
print('Column names:')
print(list(final_df.columns))

aliases = {
    'year': ['Year'],
    'player_name': ['Player_Name', 'PlayerName'],
    'team_initials': ['Team_Initials', 'TeamInitials'],
    'home_team': ['Home_Team', 'Home_Team_Name'],
    'away_team': ['Away_Team', 'Away_Team_Name'],
    'team_goals_for': ['Team_Goals_For', 'Home_Team_Goals', 'Home_Goals'],
    'team_goals_against': ['Team_Goals_Against', 'Away_Team_Goals', 'Away_Goals'],
    'total_goals': ['Total_Goals'],
    'goal_diff': ['Goal_Diff'],
    'stage': ['Stage_Std', 'Stage'],
    'winner': ['Tournament_Winner', 'Winner'],
    'attendance': ['Attendance'],
}

col_map = {k: first_existing(final_df, v) for k, v in aliases.items()}
for key, value in col_map.items():
    print(f'{key}: {value}')

if col_map['year']:
    print('\nTop tournament years by row count:')
    display(final_df[col_map['year']].value_counts().sort_index().to_frame('rows'))

if col_map['home_team'] and col_map['away_team']:
    teams = pd.concat([final_df[col_map['home_team']], final_df[col_map['away_team']]], ignore_index=True).dropna()
    print('\nTop teams by appearances:')
    display(teams.value_counts().head(15).to_frame('appearances'))

if col_map['team_goals_for'] and col_map['team_goals_against']:
    goals_for = pd.to_numeric(final_df[col_map['team_goals_for']], errors='coerce').fillna(0)
    goals_against = pd.to_numeric(final_df[col_map['team_goals_against']], errors='coerce').fillna(0)
    goals = goals_for + goals_against
    print('\nGoals summary:')
    print(goals.describe().round(2))

if col_map['attendance']:
    attendance = pd.to_numeric(final_df[col_map['attendance']], errors='coerce')
    print('\nAttendance summary:')
    print(attendance.describe().round(2))

if col_map['stage']:
    print('\nStage distribution:')
    display(final_df[col_map['stage']].value_counts().to_frame('rows'))

if col_map['winner']:
    print('\nWinner frequency:')
    display(final_df[col_map['winner']].value_counts().head(10).to_frame('rows'))


Final combined file: /Users/aarsh.user/Desktop/Major Project  (for internship)/D_V_A_/SectionE_g11_FIFA/data/processed/wc_players_combined.csv
Rows: 37,048 | Columns: 57 | Size: 11540.3 KB
Column names:
['Year', 'MatchID', 'RoundID', 'Match_Date', 'Match_Time', 'Day_of_Week', 'Month', 'Stage_Std', 'Stage_Order', 'Stage_Raw', 'Days_Into_Tournament', 'Rest_Days', 'Stadium', 'City', 'Host_Country', 'Home_Team', 'Away_Team', 'Tournament_Host', 'Is_Host_Home', 'Is_Host_Away', 'Player_Team', 'Team_Initials', 'Player_Name', 'Shirt_Number', 'Coach', 'Is_Starter', 'Is_Substitute', 'Is_Captain', 'Is_GK', 'Is_Home_Team', 'Is_Host_Team', 'Goals', 'Own_Goals', 'Goals_Fully_Attributed', 'Yellow_Cards', 'Red_Cards', 'Second_Yellow_Red', 'Effective_Red_Cards', 'Suspended_Next_Match', 'Penalties_Scored', 'Missed_Penalties', 'Subbed_In', 'Subbed_Out', 'Match_Result', 'Win_Conditions', 'Team_Won', 'Team_Goals_For', 'Team_Goals_Against', 'Total_Goals', 'Goal_Diff', 'HT_Home_Goals', 'HT_Away_Goals', 'Atten

,rows
Year,
1930,687
1934,702
1938,762
1950,918
1954,1140
1958,1540
1962,1408
1966,1408
1970,1399



Top teams by appearances:


,appearances
Brazil,4607
Italy,3675
Argentina,3402
England,2754
West Germany,2726
Spain,2619
France,2606
Mexico,2337
Uruguay,2254
Netherlands,2237



Goals summary:
count    37048.00
mean         2.83
std          1.95
min          0.00
25%          1.00
50%          3.00
75%          4.00
max         12.00
dtype: float64

Attendance summary:
count     37048.00
mean      45102.80
std       23335.33
min        2000.00
25%       30043.00
50%       41300.00
75%       61112.00
max      173850.00
Name: Attendance, dtype: float64

Stage distribution:


,rows
Stage_Std,
Group Stage,28281
Round of 16,2881
Quarter-finals,2747
Semi-finals,1498
Final,841
Third Place,800



Winner frequency:


,rows
Tournament_Winner,
Brazil,9579
Italy,6688
West Germany,5105
Argentina,3960
Germany,2944
Spain,2943
France,2816
Uruguay,1605
England,1408


## 4) Statistical analysis

This section summarizes the main analytical indicators used in the project.

In [20]:
analysis_rows = []

if col_map.get('year') and 'MatchID' in final_df.columns:
    analysis_rows.append(('Tournaments covered', int(final_df[col_map['year']].nunique())))
    analysis_rows.append(('Unique matches', int(final_df['MatchID'].nunique())))
    analysis_rows.append(('Mean rows per tournament', round(final_df.groupby(col_map['year']).size().mean(), 2)))

if col_map.get('team_goals_for') and col_map.get('team_goals_against'):
    goals_for = pd.to_numeric(final_df[col_map['team_goals_for']], errors='coerce').fillna(0)
    goals_against = pd.to_numeric(final_df[col_map['team_goals_against']], errors='coerce').fillna(0)
    total_goals = goals_for + goals_against
    analysis_rows.append(('Average total goals per row', round(total_goals.mean(), 2)))
    analysis_rows.append(('Median total goals per row', round(total_goals.median(), 2)))
    analysis_rows.append(('Max total goals per row', round(total_goals.max(), 2)))

if col_map.get('attendance'):
    attendance = pd.to_numeric(final_df[col_map['attendance']], errors='coerce')
    analysis_rows.append(('Average attendance', round(attendance.mean(), 2)))
    analysis_rows.append(('Median attendance', round(attendance.median(), 2)))

if col_map.get('home_team') and col_map.get('away_team'):
    team_rows = pd.concat([final_df[col_map['home_team']], final_df[col_map['away_team']]], ignore_index=True).dropna()
    analysis_rows.append(('Unique team labels', int(team_rows.nunique())))
    analysis_rows.append(('Top team appearances', int(team_rows.value_counts().iloc[0])))

analysis_df = pd.DataFrame(analysis_rows, columns=['metric', 'value'])
display(analysis_df)

if col_map.get('stage') and col_map.get('team_goals_for') and col_map.get('team_goals_against'):
    stage_summary = final_df.groupby(col_map['stage'])[[col_map['team_goals_for'], col_map['team_goals_against']]].mean().round(2)
    display(stage_summary)

if col_map.get('winner') and col_map.get('year'):
    champions = final_df[[col_map['year'], col_map['winner']]].dropna().drop_duplicates().sort_values(col_map['year'])
    print('Champions over time:')
    display(champions.tail(20))


,metric,value
0,Tournaments covered,20.00
1,Unique matches,836.00
2,Mean rows per tournament,1852.40
3,Average total goals per row,2.83
4,Median total goals per row,3.00
5,Max total goals per row,12.00
6,Average attendance,45102.80
7,Median attendance,41300.00
8,Unique team labels,82.00
9,Top team appearances,4607.00


,Team_Goals_For,Team_Goals_Against
Stage_Std,,
Final,1.78,1.78
Group Stage,1.39,1.38
Quarter-finals,1.41,1.39
Round of 16,1.28,1.28
Semi-finals,1.80,1.77
Third Place,1.97,1.97


Champions over time:


,Year,Tournament_Winner
0,1930,Uruguay
687,1934,Italy
1389,1938,Italy
2151,1950,Uruguay
3069,1954,West Germany
4209,1958,Brazil
5749,1962,Brazil
7157,1966,England
8565,1970,Brazil
9964,1974,West Germany


## 5) Final load and combined CSV details

This section is the final quality gate for the combined dataset.

In [29]:
combined = final_df.copy()
combined_snapshot = snapshot(combined)

display(pd.DataFrame([combined_snapshot]))

print('Combined CSV snapshot:')
print(f"- File: {final_path}")
print(f"- Rows: {combined_snapshot['rows']:,}")
print(f"- Columns: {combined_snapshot['cols']}")
print(f"- Duplicates: {combined_snapshot['dupes']}")
print(f"- Nulls: {combined_snapshot['nulls']:,}")
print(f"- Memory: {combined_snapshot['memory_mb']} MB")

print('\nNulls by column (top 20):')
nulls = combined.isna().sum().sort_values(ascending=False)
display(nulls[nulls > 0].head(20).to_frame('null_count'))

key_candidates = ['Year', 'MatchID', 'RoundID', 'Player_Name', 'Team_Initials', 'Home_Team', 'Away_Team', 'Winner', 'Attendance']
key_cols = [c for c in key_candidates if c in combined.columns]
if key_cols:
    key_quality = pd.DataFrame({
        'column': key_cols,
        'nulls': [int(combined[c].isna().sum()) for c in key_cols],
        'unique_values': [int(combined[c].nunique(dropna=True)) for c in key_cols],
    })
    display(key_quality)

readiness_checks = {
    'rows >= 10,000': combined_snapshot['rows'] >= 10_000,
    'columns >= 15': combined_snapshot['cols'] >= 15,
    'no duplicates': combined_snapshot['dupes'] == 0,
    'final file exists': final_path.exists(),
    'has core join keys': all(c in combined.columns for c in [c for c in ['Year', 'MatchID', 'RoundID'] if c]),
}

readiness_score = round(sum(readiness_checks.values()) / len(readiness_checks) * 100, 1)
display(pd.DataFrame([{'check': k, 'passed': v} for k, v in readiness_checks.items()]))
print(f'Readiness score: {readiness_score}%')


,rows,cols,nulls,dupes,memory_mb
0,37048,57,12436,0,50.18


Combined CSV snapshot:
- File: /Users/aarsh.user/Desktop/Major Project  (for internship)/D_V_A_/SectionE_g11_FIFA/data/processed/wc_players_combined.csv
- Rows: 37,048
- Columns: 57
- Duplicates: 0
- Nulls: 12,436
- Memory: 50.18 MB

Nulls by column (top 20):


,null_count
Rest_Days,9367
Shirt_Number,3069


,column,nulls,unique_values
0,Year,0,20
1,MatchID,0,836
2,RoundID,0,101
3,Player_Name,0,7638
4,Team_Initials,0,82
5,Home_Team,0,77
6,Away_Team,0,82
7,Attendance,0,622


,check,passed
0,"rows >= 10,000",True
1,columns >= 15,True
2,no duplicates,True
3,final file exists,True
4,has core join keys,True


Readiness score: 100.0%


## Summary

Use this notebook as the one-stop report for extraction, cleaning, EDA, analysis, and final combined CSV validation.

If you want, the next step can be to turn this notebook into an automated dashboard-style report with charts and exports.